# IFRS 9 Expected Credit Loss (ECL) Model
## Notebook 2 — Probability of Default (PD) Model

This notebook covers the PD component of IFRS 9 ECL:
- Through-the-Cycle (TTC) PD by rating grade
- 12-month PD (Stage 1) vs Lifetime PD (Stage 2/3)
- Survival function approach for lifetime PD
- Macroeconomic overlay (Point-in-Time adjustment)
- PD term structure by rating grade

**IFRS 9 Requirement:** PD estimates must be forward-looking and incorporate
macroeconomic information (IFRS 9.B5.5.49).

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import load_loan_portfolio, assign_stage
from src.pd_model import PDModel, TTC_ANNUAL_PD

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df = assign_stage(load_loan_portfolio())
pd_model = PDModel(macro_scalar=1.0)  # base scenario
df = pd_model.assign_pd(df)

print(f'Portfolio: {len(df):,} loans')
print(df[['stage','pd_12m','pd_lifetime','pd_applied']].groupby('stage').mean().round(4))

## 1. TTC PD Scale by Rating Grade

In [ ]:
rating_labels = {1:'AAA',2:'AA',3:'A',4:'BBB',5:'BB',6:'B',7:'CCC',8:'D'}
ratings = list(TTC_ANNUAL_PD.keys())
pds = list(TTC_ANNUAL_PD.values())

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#27ae60','#2ecc71','#f1c40f','#e67e22','#e74c3c','#c0392b','#8e44ad','#2c3e50']
bars = ax.bar([rating_labels[r] for r in ratings], [p * 100 for p in pds],
               color=colors, alpha=0.85, edgecolor='white')

for bar, pd_val in zip(bars, pds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pd_val:.1%}', ha='center', fontsize=9)

ax.set_title('Through-the-Cycle (TTC) Annual PD by Rating Grade', fontweight='bold')
ax.set_ylabel('Annual PD (%)')
ax.set_xlabel('Rating Grade')
plt.tight_layout()
plt.savefig('../plots/02_ttc_pd_scale.png', bbox_inches='tight')
plt.show()

## 2. PD Term Structure by Rating Grade

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

selected_ratings = [3, 4, 5, 6, 7]
palette = ['#27ae60','#f1c40f','#e67e22','#e74c3c','#8e44ad']

for rating, color in zip(selected_ratings, palette):
    ts = pd_model.pd_term_structure(rating, max_years=10)
    label = f'Rating {rating} ({rating_labels[rating]})'
    axes[0].plot(ts['year'], ts['marginal_pd'] * 100, 'o-', color=color, label=label)
    axes[1].plot(ts['year'], ts['cumulative_pd'] * 100, 'o-', color=color, label=label)

axes[0].set_title('Marginal PD Term Structure')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Marginal PD (%)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.4)

axes[1].set_title('Cumulative PD Term Structure')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Cumulative PD (%)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.4)

plt.suptitle('PD Term Structure by Rating Grade (Base Scenario)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/02_pd_term_structure.png', bbox_inches='tight')
plt.show()

## 3. Macroeconomic Overlay — Scenario Comparison

In [ ]:
scenarios = {'Benign (×0.7)': 0.70, 'Base (×1.0)': 1.00, 'Adverse (×1.8)': 1.80}
scenario_colors = ['#27ae60', '#2980b9', '#e74c3c']

fig, ax = plt.subplots(figsize=(12, 5))

for (label, scalar), color in zip(scenarios.items(), scenario_colors):
    m = PDModel(macro_scalar=scalar)
    ts = m.pd_term_structure(5, max_years=10)  # BBB rating
    ax.plot(ts['year'], ts['cumulative_pd'] * 100, 'o-', color=color, label=label, lw=2)

ax.set_title('Lifetime Cumulative PD — Rating BBB (Macro Scenario Comparison)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative PD (%)')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('../plots/02_pd_scenario_comparison.png', bbox_inches='tight')
plt.show()

# Portfolio-level PD comparison
print('=== Portfolio Average PD by Scenario ===')
for label, scalar in scenarios.items():
    m = PDModel(macro_scalar=scalar)
    df_s = m.assign_pd(df)
    print(f'{label}: avg PD_applied = {df_s["pd_applied"].mean():.4%}')

## 4. 12-Month vs Lifetime PD Distribution by Stage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
stage_colors = {1: '#2196F3', 2: '#FF9800', 3: '#F44336'}

for stage, color in stage_colors.items():
    subset = df[df['stage'] == stage]
    axes[0].hist(subset['pd_12m'], bins=30, alpha=0.6, color=color,
                 label=f'Stage {stage}', density=True)
    if stage in [2, 3]:
        axes[1].hist(subset['pd_lifetime'], bins=30, alpha=0.6, color=color,
                     label=f'Stage {stage}', density=True)

axes[0].set_title('12-Month PD Distribution (All Stages)')
axes[0].set_xlabel('12-Month PD')
axes[0].set_ylabel('Density')
axes[0].legend()

axes[1].set_title('Lifetime PD Distribution (Stage 2 & 3 Only)')
axes[1].set_xlabel('Lifetime PD')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('PD Distribution by IFRS 9 Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/02_pd_distribution_by_stage.png', bbox_inches='tight')
plt.show()